<a href="https://colab.research.google.com/github/KinzaAsif2456/discoverey/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding A — "What Predicts Health?"

The paper identifies Average Position (43%), Impressions (32%), and Scroll Depth (15%) as the most important predictors of the Health Score using a Random Forest model
### Where does the label actually come from?

The Health Score is calculated directly from four metrics:

* Impressions (30 points)
* Average Position (30 points)
* CTR (20 points)
* Scroll Depth (20 points)

That definition matters.Three of the highest-ranked features are already part of the target itself. So the model is mostly picking up the same structure that was already used to calculate the Health Score. If impressions contribute directly to the Health Score, it is expected that they will receive high feature importance.

So this feature importance tells us more about how the Health Score was built than about what actually causes a website to be healthy

### Does the validation support the claim?

The paper does say that this feature importance is descriptive and not proof of causation. I think the more fundamental limitation is that validation cannot solve this issue. Even if the paper used a different validation split, these variables would probably still have high importance because they are already used to calculate the target.

The analysis is still useful for showing how the scoring system behaves, but it does not really tell us which factors independently affect website health




## Finding B — "What Predicts Growth?"

The paper trains a Logistic Regression model to classify pages as growing or declining and reports 71% accuracy from a single 80/20 train-test split. Content Age receives the largest coefficient.

### Where does the label come from?

The target compares performance during the most recent 30 days with the previous 30 days. My capstone uses the same idea through the 'is_declining_label', so I already knew that evaluation would be just as important as model selection.

Since the label represents a change over time, the evaluation strategy becomes especially important.Because the label is based on change over time, I think the model should also be tested in a way that is closer to how it would be used on future data, instead of relying on one random split.

### Does the validation support the claim?

Here the limitation is different.

The paper reports only one holdout split and does not explain whether pages from the same client were kept together or whether time ordering was respected. Because these details are not given, I can't tell if the 71% accuracy would stay similar on other data.

The paper also says that Content Age can affect the comparison between models, but Content Age still has the strongest coefficient. So I am not completely sure whether the model is learning general patterns or mainly relying on page age. It is possible that the model is relying too much on page age instead of finding patterns that would also work on future data.

During my own evaluation, I used GroupKFold instead of relying on a single split. Precision@50 changed noticeably across folds, even though the average performance looked reasonable. That experience showed me how sensitive the results were to the evaluation strategy.

Since the paper only gives one holdout result, I can't tell if 71% is a stable result or if that particular split just happened to work well. Testing it across several grouped splits would make this result easier to trust.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
#IMPORTS

import numpy as np
import pandas as pd
import os
import subprocess
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

In [ ]:
# Step 1 — Get the Dataset

STARTER_REPO = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
if not os.path.isdir("flyrank-ml-internship-starter"):
    subprocess.run(["git", "clone", "--depth", "1", STARTER_REPO, "flyrank-ml-internship-starter"], check=True)


df = pd.read_csv("flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv")


In [ ]:
# Step 2 —  Basic dataset checks
print(f"Dataset shape: {df.shape}")
print(f"Duplicate content_ids: {df['content_id'].duplicated().sum()}")
print(f"Overall decline rate: {df['trend_direction'].eq('down').mean():.3f}")
print("duplicate content_id rows:", df["content_id"].duplicated().sum())

Dataset shape: (30000, 44)
Duplicate content_ids: 0
Overall decline rate: 0.542
duplicate content_id rows: 0


In [ ]:

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

tier_order = ["0-30", "31-90", "91-180", "181+"]
df["freshness_tier_enc"] = df["freshness_tier"].map({t: i for i, t in enumerate(tier_order)})
df["avg_position_missing"] = (df["avg_position"] == 0).astype(int)

FEATURES = ["freshness_tier_enc", "avg_position", "ctr", "impressions_90d", "avg_position_missing"]

def precision_at_k(sub_df, score_col, k=50, tiebreak_col="impressions_90d"):
    top_k = sub_df.sort_values([score_col, tiebreak_col], ascending=[False, False]).head(k)
    return top_k["is_declining_label"].mean()

def make_rf():
    return RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=5,
                                   random_state=42, n_jobs=-1)

In [ ]:
# BEFORE — Random Split
random_p50s = []
for seed in [0, 1, 42, 100, 7]:
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=seed,
                                          stratify=df["is_declining_label"])
    rf = make_rf()
    rf.fit(train_df[FEATURES], train_df["is_declining_label"])
    test_df = test_df.copy()
    test_df["score_rf"] = rf.predict_proba(test_df[FEATURES])[:, 1]
    random_p50s.append(precision_at_k(test_df, "score_rf"))

In [ ]:
# AFTER — Grouped split by client
gkf = GroupKFold(n_splits=5)
grouped_p50s = []
for fold, (tr_idx, te_idx) in enumerate(gkf.split(df, groups=df["client_id"])):
    train_fold, test_fold = df.iloc[tr_idx], df.iloc[te_idx].copy()
    rf = make_rf()
    rf.fit(train_fold[FEATURES], train_fold["is_declining_label"])
    test_fold["score_rf"] = rf.predict_proba(test_fold[FEATURES])[:, 1]
    grouped_p50s.append(precision_at_k(test_fold, "score_rf"))

In [ ]:

print(f"BEFORE (random split):  {np.mean(random_p50s):.3f} ± {np.std(random_p50s):.3f}")
print(f"AFTER  (grouped split): {np.mean(grouped_p50s):.3f} ± {np.std(grouped_p50s):.3f}")

BEFORE (random split):  0.880 ± 0.031
AFTER  (grouped split): 0.872 ± 0.069


### Effect of Using a Grouped Split

When I compared the two evaluation methods, the average Precision@50 changed only slightly. The random split achieved **0.880 ± 0.031**, while the grouped split achieved **0.872 ± 0.069**.The difference between the two mean scores is only 0.008, so I did not observe a large drop in performance when I changed from a random split to a grouped split.

I think this is because my features do not contain any obvious client-specific information. The model uses features such as `freshness_tier_enc`, `avg_position`, `ctr`, `impressions_90d`, and `avg_position_missing`, none of which identify a particular client. Because these features do not identify the client, the model has less information that it could use to memorize individual clients.

The bigger difference was in the variability. The standard deviation increased from **±0.031** with the random split to **±0.069** with the grouped split. This means the model's performance changes more depending on which clients end up in each fold. Given that the dataset contains only **32 clients**, different groupings can produce noticeably different results. So even though the average score stayed almost the same, the grouped split showed that the model is less consistent across different client groups.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
BANNED = ["trend_direction", "trend_pct"]
PRODUCT_FLAGS = ["stale_flag", "low_ctr_flag", "is_decoy", "baseline_score"]
window_terms = ["last_30d", "prev_30d", "prev30"]

print("Check 1 — banned/label-derived:", [c for c in FEATURES if c in BANNED] or "clean")
print("Check 2 — product flags:", [c for c in FEATURES if c in PRODUCT_FLAGS] or "clean")
print("Check 3 — overlapping windows:", [c for c in FEATURES if any(t in c for t in window_terms)] or "clean")

Check 1 — banned/label-derived: clean
Check 2 — product flags: clean
Check 3 — overlapping windows: clean


In [ ]:


# Check 4 — prove the test harness actually catches leakage
df["_leak_probe"] = df["trend_pct"]
LEAKY_FEATURES = FEATURES + ["_leak_probe"]

leak_p50s = []
for fold, (tr_idx, te_idx) in enumerate(gkf.split(df, groups=df["client_id"])):
    train_fold, test_fold = df.iloc[tr_idx], df.iloc[te_idx].copy()
    rf = make_rf()
    rf.fit(train_fold[LEAKY_FEATURES], train_fold["is_declining_label"])
    test_fold = test_fold.copy()
    test_fold["score_leak"] = rf.predict_proba(test_fold[LEAKY_FEATURES])[:, 1]
    leak_p50s.append(precision_at_k(test_fold, "score_leak"))



In [ ]:
print(f"\nHonest feature set:      P@50 = {np.mean(grouped_p50s):.3f}")
print(f"+ trend_pct injected:    P@50 = {np.mean(leak_p50s):.3f}")


Honest feature set:      P@50 = 0.872
+ trend_pct injected:    P@50 = 1.000


These checks look for the main leakage patterns I expected from the feature names and how the features were created. The injection test gives me an extra check: if I deliberately add target information, the model's score should change.

**Important:** The **1.000 Precision@50 is not a real model result**. I got this score by intentionally adding `trend_pct`, which contains information about the target. I only did this to check whether my evaluation would react to leakage. The actual model results use the original feature set without `trend_pct`.



In [ ]:
df.drop(columns=["_leak_probe"], inplace=True)

### Leakage Checks

The original feature set passed the three direct leakage checks, and the grouped evaluation gave a **P@50 of 0.872**. I then added `trend_pct` on purpose to see if the leakage check would actually catch it. The score went up to **1.000**, which is what I expected because this feature contains direct target information. I did not use this leaked result as the model's performance.



## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# 1. Create target


df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

In [ ]:
# 2. Recreate rule-based baseline


STALE_TIERS = ["91-180", "181+"]
CTR_THRESHOLD = 0.005
IMPRESSION_DECOY_LEVEL = 5000

df["stale_flag"] = df["freshness_tier"].isin(STALE_TIERS)

df["low_ctr_flag"] = (
    (df["avg_position"] <= 20)
    & (df["ctr"] < CTR_THRESHOLD)
    & (df["impressions_90d"] >= 500)
)

df["is_decoy"] = (
    (df["freshness_tier"] == "181+")
    & (df["impressions_90d"] >= IMPRESSION_DECOY_LEVEL)
)


def calculate_score(row):
    if row["is_decoy"]:
        return 3
    if row["stale_flag"] and row["low_ctr_flag"]:
        return 2
    if row["stale_flag"] or row["low_ctr_flag"]:
        return 1
    return 0


df["baseline_score"] = df.apply(
    calculate_score,
    axis=1
)



In [ ]:
# 3. Create model features


tier_order = ["0-30", "31-90", "91-180", "181+"]

df["freshness_tier_enc"] = df["freshness_tier"].map(
    {tier: i for i, tier in enumerate(tier_order)}
)

df["avg_position_missing"] = (
    df["avg_position"] == 0
).astype(int)


FEATURES = [
    "freshness_tier_enc",
    "avg_position",
    "ctr",
    "impressions_90d",
    "avg_position_missing"
]


In [ ]:
# 4. Precision@50


def precision_at_k(sub_df, score_col, k=50):
    ranked = sub_df.sort_values(
        [score_col, "impressions_90d"],
        ascending=[False, False]
    )

    top_k = ranked.head(k)

    return top_k["is_declining_label"].mean()

In [ ]:
# 5. Grouped 5-fold CV


gkf = GroupKFold(n_splits=5)

baseline_scores = []
lr_scores = []
rf_scores = []

lr_aucs = []
rf_aucs = []


for fold, (train_idx, test_idx) in enumerate(
    gkf.split(df, groups=df["client_id"])
):

    train_fold = df.iloc[train_idx].copy()
    test_fold = df.iloc[test_idx].copy()

    y_train = train_fold["is_declining_label"]
    y_test = test_fold["is_declining_label"]


    # Baseline
    baseline_p50 = precision_at_k(
        test_fold,
        "baseline_score"
    )

    baseline_scores.append(baseline_p50)


    # Logistic Regression
    lr_model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        ))
    ])

    lr_model.fit(
        train_fold[FEATURES],
        y_train
    )

    lr_prob = lr_model.predict_proba(
        test_fold[FEATURES]
    )[:, 1]

    test_fold["lr_score"] = lr_prob

    lr_scores.append(
        precision_at_k(
            test_fold,
            "lr_score"
        )
    )

    lr_aucs.append(
        roc_auc_score(
            y_test,
            lr_prob
        )
    )


    # Random Forest
    rf_model = RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )

    rf_model.fit(
        train_fold[FEATURES],
        y_train
    )

    rf_prob = rf_model.predict_proba(
        test_fold[FEATURES]
    )[:, 1]

    test_fold["rf_score"] = rf_prob

    rf_scores.append(
        precision_at_k(
            test_fold,
            "rf_score"
        )
    )

    rf_aucs.append(
        roc_auc_score(
            y_test,
            rf_prob
        )
    )


    print(
        f"Fold {fold + 1}: "
        f"Baseline={baseline_scores[-1]:.3f}, "
        f"LR={lr_scores[-1]:.3f}, "
        f"RF={rf_scores[-1]:.3f}"
    )


Fold 1: Baseline=0.740, LR=0.920, RF=0.940
Fold 2: Baseline=0.960, LR=0.740, RF=0.800
Fold 3: Baseline=0.620, LR=0.580, RF=0.900
Fold 4: Baseline=0.900, LR=0.760, RF=0.940
Fold 5: Baseline=0.720, LR=0.400, RF=0.780


In [ ]:
# 6. Comparison table


comparison_df = pd.DataFrame({
    "Method": [
        "Rule-based Baseline",
        "Improved Logistic Regression",
        "Random Forest"
    ],

    "Precision@50": [
        f"{np.mean(baseline_scores):.3f} ± {np.std(baseline_scores):.3f}",
        f"{np.mean(lr_scores):.3f} ± {np.std(lr_scores):.3f}",
        f"{np.mean(rf_scores):.3f} ± {np.std(rf_scores):.3f}"
    ],

    "ROC-AUC": [
        "N/A",
        f"{np.mean(lr_aucs):.3f} ± {np.std(lr_aucs):.3f}",
        f"{np.mean(rf_aucs):.3f} ± {np.std(rf_aucs):.3f}"
    ]
})

display(comparison_df)

,Method,Precision@50,ROC-AUC
0,Rule-based Baseline,0.788 ± 0.124,N/A
1,Improved Logistic Regression,0.680 ± 0.177,0.580 ± 0.064
2,Random Forest,0.872 ± 0.069,0.665 ± 0.045


### Evidence and interpretation


On this dataset, the Random Forest had the highest **observed Precision@50** in the 5-fold grouped evaluation, with **0.872 ± 0.069**, compared with **0.788 ± 0.124** for the rule-based baseline and **0.680 ± 0.177** for the improved Logistic Regression.Since the scores were different across folds, I would treat this as a directional result rather than saying the Random Forest will always perform best on new data

The Random Forest performed better than both the baseline and Logistic Regression in these folds. This may be because the Random Forest can handle the threshold-like patterns in the data better. However, I would not claim that other models such as XGBoost or LightGBM would perform better, because I did not test them.



In [ ]:
# Real failure examples from the grouped evaluation

failure_rows = []

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(df, groups=df["client_id"])
):
    train_fold = df.iloc[train_idx].copy()
    test_fold = df.iloc[test_idx].copy()

    y_train = train_fold["is_declining_label"]

    # # Train Logistic Regression
    lr_model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        ))
    ])

    lr_model.fit(
        train_fold[FEATURES],
        y_train
    )

    # Model probability
    test_fold["lr_score"] = lr_model.predict_proba(
        test_fold[FEATURES]
    )[:, 1]

    # Select the model's Top-50
    ranked = test_fold.sort_values(
        ["lr_score", "impressions_90d"],
        ascending=[False, False]
    )

    top_50_ids = set(ranked.head(50)["content_id"])

    test_fold["in_top50"] = (
        test_fold["content_id"].isin(top_50_ids)
    )

    test_fold["fold"] = fold + 1

    failure_rows.append(test_fold)


# Combine all grouped test folds
failure_df = pd.concat(
    failure_rows,
    ignore_index=True
)

# False positives:
# pages ranked in the Top-50 but not actually declining
false_positives = failure_df[
    (failure_df["in_top50"]) &
    (failure_df["is_declining_label"] == 0)
].copy()

# Missed decliners:
# Declining pages missed by the model
missed_decliners = failure_df[
    (~failure_df["in_top50"]) &
    (failure_df["is_declining_label"] == 1)
].copy()

print("False positives:", len(false_positives))
print("Missed decliners:", len(missed_decliners))

print("\n=== Top False Positives ===")

display(
    false_positives[
        [
            "content_id",
            "client_id",
            "freshness_tier",
            "avg_position",
            "ctr",
            "impressions_90d",
            "lr_score",
            "is_declining_label"
        ]
    ]
    .sort_values("lr_score", ascending=False)
    .head(5)
)

print("\n=== Sample Missed Decliners ===")

display(
    missed_decliners[
        [
            "content_id",
            "client_id",
            "freshness_tier",
            "avg_position",
            "ctr",
            "impressions_90d",
            "lr_score",
            "is_declining_label"
        ]
    ]
    .sort_values("lr_score", ascending=False)
    .head(5)
)

False positives: 80
Missed decliners: 16092

=== Top False Positives ===


,content_id,client_id,freshness_tier,avg_position,ctr,impressions_90d,lr_score,is_declining_label
27257,content_06e19c6486b0,client_4ec9599fc2,181+,5.0,0.0,10,0.701904,0
24489,content_ab27c30d81f4,client_4ec9599fc2,181+,8.9,0.0,103,0.691203,0
29785,content_b51d84226fc9,client_9f14025af0,91-180,1.0,0.0,1,0.664714,0
29555,content_4d1ebe33b02d,client_9f14025af0,91-180,1.0,0.0,1,0.664714,0
27260,content_201a4a56f4d6,client_8527a891e2,91-180,1.0,0.0,2,0.664712,0



=== Sample Missed Decliners ===


,content_id,client_id,freshness_tier,avg_position,ctr,impressions_90d,lr_score,is_declining_label
27628,content_7b06c3ec3b03,client_4ec9599fc2,91-180,4.4,0.00,35,0.654907,1
25337,content_fe16a55cd13d,client_7f2253d7e2,181+,16.4,0.33,4556,0.654845,1
29671,content_c6cffac60462,client_8527a891e2,91-180,4.6,0.00,39,0.654322,1
25970,content_895f8188c254,client_4ec9599fc2,91-180,4.6,0.00,123,0.654142,1
24603,content_c3e4ddfb8650,client_8527a891e2,91-180,4.7,0.00,7,0.654102,1


## **Rewritten Claim**

**On this dataset, the Random Forest had the highest observed Precision@50 in the grouped 5-fold evaluation. It could be useful for prioritizing pages that may be declining, but this is only a directional result. I would want to test it on more clients or future data before relying on it in practice.**



### What the errors show

The false positives show where Logistic Regression has problems. Many of the pages it ranked highly were old pages with very low impressions and zero CTR. For example, one page had only 10 impressions and another had 103, but both received high model scores. This suggests that the model is using age, position, and CTR, but it is not applying the traffic threshold in the same way as the rule-based baseline.

The missed decliners show the other side of the problem. Some declining pages had much higher traffic, so the model did not rank them in the Top-50. So the model is not just picking every old or low-traffic page. It also misses some real decliners when their other features look different from the patterns it learned.

Overall, these examples help explain why Logistic Regression had a lower Precision@50 than Random Forest. The errors also give a possible reason for this difference. Random Forest can handle non-linear relationships and interactions between features, while Logistic Regression is more limited because it is a linear model.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.